# **Examen Segundo Bimestre**  

---
- Nombre: Michael Enríquez
- Fecha:
---

### Preparación del corpus.

In [16]:
import kagglehub
import os
import numpy as np
import pandas as pd

from dotenv import load_dotenv
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer, CrossEncoder

In [17]:
# Download latest version
path = kagglehub.dataset_download("spsayakpaul/arxiv-paper-abstracts")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'arxiv-paper-abstracts' dataset.
Path to dataset files: /kaggle/input/arxiv-paper-abstracts


In [18]:
csv_path = os.path.join(path, "arxiv_data.csv")  # <-- cambiar si el nombre difiere

df = pd.read_csv(csv_path)
print(df.shape)
df.head()

(51774, 3)


,titles,summaries,terms
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']"
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']"
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","['cs.CV', 'cs.AI']"
3,Parameter Decoupling Strategy for Semi-supervi...,Consistency training has proven to be an advan...,['cs.CV']
4,Background-Foreground Segmentation for Interio...,"To ensure safety in automated driving, the cor...","['cs.CV', 'cs.LG']"


In [19]:
# Eliminar filas con valores nulos en columnas clave
df = df.dropna(subset=["titles", "summaries", "terms"])  # <-- ajustar nombres según columnas reales

# Eliminar duplicados exactos
df = df.drop_duplicates(subset=["titles", "summaries"])

df = df.reset_index(drop=True)
print("Shape final:", df.shape)

Shape final: (38985, 3)


In [20]:
SAMPLE_SIZE = 8000  # ajusta según lo que necesites

df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)
print("Shape tras muestreo:", df.shape)

Shape tras muestreo: (8000, 3)


In [21]:
df["document"] = (
    "Title: " + df["titles"] +
    "\nAbstract: " + df["summaries"] +
    "\nTopics: " + df["terms"].astype(str)
)

print(df["document"].iloc[0])

Title: A Three-stage Approach for Segmenting Degraded Color Images: Smoothing, Lifting and Thresholding (SLaT)
Abstract: In this paper, we propose a SLaT (Smoothing, Lifting and Thresholding) method
with three stages for multiphase segmentation of color images corrupted by
different degradations: noise, information loss, and blur. At the first stage,
a convex variant of the Mumford-Shah model is applied to each channel to obtain
a smooth image. We show that the model has unique solution under the different
degradations. In order to properly handle the color information, the second
stage is dimension lifting where we consider a new vector-valued image composed
of the restored image and its transform in the secondary color space with
additional information. This ensures that even if the first color space has
highly correlated channels, we can still have enough information to give good
segmentation results. In the last stage, we apply multichannel thresholding to
the combined vector-value

---
### Fase 2: Representación mediante Embeddings.

In [22]:
!pip install -q -U sentence-transformers

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Usando device:", device)

embedding_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5",
    device=device
)

In [24]:
documents = df["document"].tolist()
print("Total de documentos a codificar:", len(documents))

Total de documentos a codificar: 8000


In [ ]:
embeddings = embedding_model.encode(
    documents,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # importante: para poder usar similitud coseno luego
)

print("Shape de embeddings:", embeddings.shape)

In [26]:
print("Dtype:", embeddings.dtype)
print("Ejemplo de vector (primeras 5 dims):", embeddings[0][:5])
print("Norma del primer vector (debería ser ~1.0):", np.linalg.norm(embeddings[0]))

Dtype: float32
Ejemplo de vector (primeras 5 dims): [-0.00440498 -0.01110226 -0.00013402  0.02935163  0.05457511]
Norma del primer vector (debería ser ~1.0): 1.0


---
### Fase 3: Almacenamiento y Búsqueda Vectorial (Milvus Lite).

In [ ]:
!pip install -q pymilvus milvus-lite

In [28]:
from pymilvus import connections, utility, FieldSchema, CollectionSchema, DataType, Collection

# Milvus Lite crea un archivo local (arxiv.db) para persistir los datos
connections.connect(alias="default", uri="./arxiv.db")

print("Conexión establecida.")
print("Colecciones existentes:", utility.list_collections())

Conexión establecida.
Colecciones existentes: []


/tmp/ipykernel_946/3663990673.py:4: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(alias="default", uri="./arxiv.db")
/tmp/ipykernel_946/3663990673.py:7: PyMilvusDeprecationWarning: `utility.list_collections` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  print("Colecciones existentes:", utility.list_collections())


In [29]:
EMBEDDING_DIM = embeddings.shape[1]  # debería ser 768 para bge-base-en-v1.5
print("Dimensión de embeddings:", EMBEDDING_DIM)

fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=EMBEDDING_DIM),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535),
]

schema = CollectionSchema(
    fields=fields,
    description="Colección de papers de arXiv con embeddings BGE"
)

Dimensión de embeddings: 768


In [30]:
COLLECTION_NAME = "papers"

if utility.has_collection(COLLECTION_NAME):
    utility.drop_collection(COLLECTION_NAME)
    print(f"Colección '{COLLECTION_NAME}' existente eliminada para recrearla limpia.")

collection = Collection(name=COLLECTION_NAME, schema=schema)
print(f"Colección '{COLLECTION_NAME}' creada.")

/tmp/ipykernel_946/321965427.py:3: PyMilvusDeprecationWarning: `utility.has_collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  if utility.has_collection(COLLECTION_NAME):
/tmp/ipykernel_946/321965427.py:7: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection = Collection(name=COLLECTION_NAME, schema=schema)


Colección 'papers' creada.


In [ ]:
BATCH_SIZE = 1000

for i in tqdm(range(0, len(documents), BATCH_SIZE)):
    batch_embeddings = embeddings[i:i+BATCH_SIZE].tolist()
    batch_texts = documents[i:i+BATCH_SIZE]

    collection.insert([batch_embeddings, batch_texts])

print("Inserción completada. Total de registros:", collection.num_entities)

In [32]:
index_params = {
    "metric_type": "COSINE",
    "index_type": "HNSW",
    "params": {"M": 16, "efConstruction": 200}
}

collection.create_index(field_name="embedding", index_params=index_params)
print("Índice HNSW creado.")

/tmp/ipykernel_946/3030004165.py:7: PyMilvusDeprecationWarning: `Collection.create_index` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.create_index(field_name="embedding", index_params=index_params)
ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1232, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


Índice HNSW creado.


In [33]:
collection.load()
print("Colección cargada y lista para búsquedas.")

Colección cargada y lista para búsquedas.


/tmp/ipykernel_946/2826397547.py:1: PyMilvusDeprecationWarning: `Collection.load` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.load()


In [34]:
test_query = "Graph Neural Networks applications"
test_embedding = embedding_model.encode(test_query, normalize_embeddings=True).tolist()

search_params = {"metric_type": "COSINE", "params": {"ef": 64}}

results = collection.search(
    data=[test_embedding],
    anns_field="embedding",
    param=search_params,
    limit=3,
    output_fields=["text"]
)

for hit in results[0]:
    print("Score:", hit.score)
    print(hit.entity.get("text")[:200])
    print("---")

/tmp/ipykernel_946/329432324.py:6: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


Score: 0.8614902496337891
Title: Graph Neural Networks: Taxonomy, Advances and Trends
Abstract: Graph neural networks provide a powerful toolkit for embedding real-world
graphs into low-dimensional spaces according to specific
---
Score: 0.8407355546951294
Title: Learning Graph Neural Networks with Approximate Gradient Descent
Abstract: The first provably efficient algorithm for learning graph neural networks
(GNNs) with one hidden layer for node inform
---
Score: 0.8370245099067688
Title: Residual or Gate? Towards Deeper Graph Neural Networks for Inductive Graph Representation Learning
Abstract: In this paper, we study the problem of node representation learning with
graph neura
---


---

### Fase 4: Recuperación.

In [35]:
# Prefijo recomendado por BGE para consultas de recuperación
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def retrieve(query, top_k=20):
    """
    Recupera los top_k documentos más similares a la query desde Milvus.

    Retorna una lista de dicts: [{"text": ..., "score": ...}, ...]
    """
    query_with_prefix = BGE_QUERY_PREFIX + query

    query_embedding = embedding_model.encode(
        query_with_prefix,
        normalize_embeddings=True
    ).tolist()

    search_params = {"metric_type": "COSINE", "params": {"ef": 64}}

    results = collection.search(
        data=[query_embedding],
        anns_field="embedding",
        param=search_params,
        limit=top_k,
        output_fields=["text"]
    )

    hits = []
    for hit in results[0]:
        hits.append({
            "text": hit.entity.get("text"),
            "score": float(hit.score)
        })

    return hits

In [36]:
test_queries = [
    "What are the main applications of Graph Neural Networks?",
    "How is reinforcement learning used in robotics?",
    "Recent advances in diffusion models for image generation.",
    "Techniques for improving retrieval-augmented generation systems."
]

sample_results = retrieve(test_queries[0], top_k=5)

for i, r in enumerate(sample_results):
    print(f"[{i+1}] Score: {r['score']:.4f}")
    print(r["text"][:200])
    print("---")

/tmp/ipykernel_946/2754236030.py:19: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


[1] Score: 0.7424
Title: Graph Neural Networks: Taxonomy, Advances and Trends
Abstract: Graph neural networks provide a powerful toolkit for embedding real-world
graphs into low-dimensional spaces according to specific
---
[2] Score: 0.7400
Title: A Practical Guide to Graph Neural Networks
Abstract: Graph neural networks (GNNs) have recently grown in popularity in the field
of artificial intelligence due to their unique ability to ingest
---
[3] Score: 0.7387
Title: Learning Graph Neural Networks with Approximate Gradient Descent
Abstract: The first provably efficient algorithm for learning graph neural networks
(GNNs) with one hidden layer for node inform
---
[4] Score: 0.7359
Title: Improving the Long-Range Performance of Gated Graph Neural Networks
Abstract: Many popular variants of graph neural networks (GNNs) that are capable of
handling multi-relational graphs may suff
---
[5] Score: 0.7345
Title: Analyzing the Performance of Graph Neural Networks with Pipe Parallelism
Abstract: 

In [37]:
for q in test_queries:
    print("="*80)
    print("QUERY:", q)
    print("="*80)
    results = retrieve(q, top_k=3)
    for r in results:
        print(f"Score: {r['score']:.4f} | {r['text'][:150]}")
    print()

QUERY: What are the main applications of Graph Neural Networks?


/tmp/ipykernel_946/2754236030.py:19: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


Score: 0.7424 | Title: Graph Neural Networks: Taxonomy, Advances and Trends
Abstract: Graph neural networks provide a powerful toolkit for embedding real-world
graphs
Score: 0.7400 | Title: A Practical Guide to Graph Neural Networks
Abstract: Graph neural networks (GNNs) have recently grown in popularity in the field
of artificial 
Score: 0.7387 | Title: Learning Graph Neural Networks with Approximate Gradient Descent
Abstract: The first provably efficient algorithm for learning graph neural net

QUERY: How is reinforcement learning used in robotics?
Score: 0.7558 | Title: Using Deep Reinforcement Learning for the Continuous Control of Robotic Arms
Abstract: Deep reinforcement learning enables algorithms to learn 
Score: 0.7519 | Title: Learning Force Control for Contact-rich Manipulation Tasks with Rigid Position-controlled Robots
Abstract: Reinforcement Learning (RL) methods 
Score: 0.7431 | Title: Creativity in Robot Manipulation with Deep Reinforcement Learning
Abstract: Deep Reinf

---

### Fase 5: Re-ranking con CrossEncoder.

In [ ]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device=device  # reutilizamos el device definido en la Fase 2 (cuda o cpu)
)

In [39]:
def rerank(query, candidates, top_k=5):
    """
    Re-ordena los candidatos recuperados usando un CrossEncoder.

    candidates: lista de dicts [{"text": ..., "score": ...}, ...] (salida de retrieve())
    Retorna: lista de dicts [{"text": ..., "rerank_score": ...}, ...] ordenada descendente
    """
    pairs = [(query, c["text"]) for c in candidates]

    rerank_scores = reranker.predict(pairs)

    reranked = [
        {"text": c["text"], "rerank_score": float(score)}
        for c, score in zip(candidates, rerank_scores)
    ]

    reranked = sorted(reranked, key=lambda x: x["rerank_score"], reverse=True)

    return reranked[:top_k]

In [40]:
def search(query, top_k_retrieve=20, top_k_final=5):
    """
    Pipeline completo de recuperación: bi-encoder (Milvus) + re-ranking (CrossEncoder).
    """
    candidates = retrieve(query, top_k=top_k_retrieve)
    final_results = rerank(query, candidates, top_k=top_k_final)
    return final_results

In [41]:
for q in test_queries:
    print("="*80)
    print("QUERY:", q)
    print("="*80)

    results = search(q)

    for i, r in enumerate(results):
        print(f"[{i+1}] Rerank score: {r['rerank_score']:.4f}")
        print(r["text"][:200])
        print("---")
    print()

QUERY: What are the main applications of Graph Neural Networks?


/tmp/ipykernel_946/2754236030.py:19: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


[1] Rerank score: 4.2091
Title: AutoGraph: Automated Graph Neural Network
Abstract: Graphs play an important role in many applications. Recently, Graph Neural
Networks (GNNs) have achieved promising results in graph analysis 
---
[2] Rerank score: 4.2030
Title: Graph Neural Networks: Taxonomy, Advances and Trends
Abstract: Graph neural networks provide a powerful toolkit for embedding real-world
graphs into low-dimensional spaces according to specific
---
[3] Rerank score: 4.1577
Title: A Practical Guide to Graph Neural Networks
Abstract: Graph neural networks (GNNs) have recently grown in popularity in the field
of artificial intelligence due to their unique ability to ingest
---
[4] Rerank score: 3.6043
Title: Graph Transformer Networks
Abstract: Graph neural networks (GNNs) have been widely used in representation learning
on graphs and achieved state-of-the-art performance in tasks such as node
clas
---
[5] Rerank score: 3.2487
Title: Higher-Order Explanations of Graph Neural Networ

---

###Fase 6: Generación aumentada por recuperación (RAG con LLM)

In [ ]:
!pip install -q -U google-genai

In [43]:
from google.colab import userdata
import os

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [44]:
from google import genai

gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

GEMINI_MODEL = "gemini-2.5-flash"

In [45]:
RAG_SYSTEM_INSTRUCTION = """Eres un asistente experto en literatura científica que responde preguntas
basándote ÚNICAMENTE en los fragmentos de evidencia proporcionados (abstracts de papers de arXiv).

Reglas estrictas:
1. Responde solo con información contenida en las evidencias.
2. Si las evidencias no contienen información suficiente para responder la consulta,
   indícalo explícitamente diciendo algo como: "El corpus no contiene información suficiente
   para responder esta consulta con certeza."
3. Si integras información de varios documentos, sé explícito al respecto.
4. No inventes datos, autores, ni resultados que no estén en las evidencias.
5. Responde en inglés si la consulta está en inglés, o en español si la consulta está en español.
"""

def build_prompt(query, evidences):
    context_blocks = []
    for i, ev in enumerate(evidences, start=1):
        context_blocks.append(f"[Evidence {i}]\n{ev['text']}")

    context_text = "\n\n".join(context_blocks)

    prompt = f"""Consulta del usuario: {query}

Evidencias recuperadas del corpus:

{context_text}

Instrucción: Responde la consulta del usuario basándote únicamente en las evidencias anteriores,
siguiendo las reglas del sistema."""

    return prompt

In [46]:
from google.genai import types
import time
import random
from google.genai import errors as genai_errors

def generate_answer(query, evidences, temperature=0.2, max_retries=5):
    prompt = build_prompt(query, evidences)

    for attempt in range(max_retries):
        try:
            response = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=RAG_SYSTEM_INSTRUCTION,
                    temperature=temperature,
                    max_output_tokens=800,
                ),
            )
            return response.text

        except genai_errors.ServerError as e:
            wait = (2 ** attempt) + random.uniform(0, 1)
            print(f"[Intento {attempt+1}/{max_retries}] Error del servidor ({e}). Reintentando en {wait:.1f}s...")
            time.sleep(wait)

    # Si se agotaron los reintentos
    return "No fue posible generar una respuesta en este momento debido a alta demanda del servicio. Por favor intenta nuevamente en unos minutos."

In [47]:
def rag_pipeline(query, top_k_retrieve=20, top_k_final=5):
    """
    Pipeline completo: retrieve -> rerank -> generate.
    Retorna la respuesta y las evidencias usadas (para la Fase 7).
    """
    evidences = search(query, top_k_retrieve=top_k_retrieve, top_k_final=top_k_final)
    answer = generate_answer(query, evidences)

    return {
        "query": query,
        "answer": answer,
        "evidences": evidences
    }

In [48]:
result = rag_pipeline(test_queries[0])

print("QUERY:", result["query"])
print("\nRESPUESTA:\n", result["answer"])

/tmp/ipykernel_946/2754236030.py:19: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


QUERY: What are the main applications of Graph Neural Networks?

RESPUESTA:
 Based on the provided evidence, the main applications of Graph Neural Networks (GNNs) include:

*   **Graph analysis tasks** (Evidence 1)
*   **Graph node classification** (Evidence 1, Evidence 4)
*   **Link prediction** (Evidence 4)
*   **Predicting graph structured data** (Evidence 5)
*   **Sentiment analysis of text data** (Evidence 5)
*   **Structure-property relationships in quantum chemistry** (Evidence 5)
*   **Image classification** (Evidence 5)


---

### Fase 7: Presentación de evidencias

In [49]:
def display_rag_result(result, max_chars=400):
    """
    Muestra de forma legible: query, respuesta generada y evidencias utilizadas.
    """
    print("="*90)
    print(f"CONSULTA: {result['query']}")
    print("="*90)

    print("\nRESPUESTA GENERADA:\n")
    print(result["answer"])

    print("\n" + "-"*90)
    print(f"EVIDENCIAS UTILIZADAS ({len(result['evidences'])}):")
    print("-"*90)

    for i, ev in enumerate(result["evidences"], start=1):
        print(f"\n[Evidencia {i}] — Rerank score: {ev['rerank_score']:.4f}")
        texto = ev["text"]
        if len(texto) > max_chars:
            texto = texto[:max_chars] + "..."
        print(texto)

    print("\n" + "="*90 + "\n")

In [50]:
result = rag_pipeline(test_queries[0])
display_rag_result(result)

/tmp/ipykernel_946/2754236030.py:19: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


CONSULTA: What are the main applications of Graph Neural Networks?

RESPUESTA GENERADA:

Las principales aplicaciones de las Redes Neuronales Gráficas (GNNs) mencionadas en el corpus son:

*   **Tareas de análisis de grafos** (Evidencia 1).
*   **Clasificación de nodos en grafos** (Evidencia 1, Evidencia 4).
*   **Predicción de enlaces** (Evidencia 4).
*   **Análisis de sentimiento de datos de texto** (Evidencia 5).
*   **Relaciones estructura-propiedad en química cuántica** (Evidencia 5).
*   **Clasificación de imágenes** (Evidencia 5).

Además, las GNNs son descritas como una herramienta poderosa para incrustar grafos del mundo real en espacios de baja dimensión según tareas específicas (Evidencia 2) y para predecir datos estructurados en grafos (Evidencia 5).

------------------------------------------------------------------------------------------
EVIDENCIAS UTILIZADAS (5):
------------------------------------------------------------------------------------------

[Evidencia 1] — 

---

### Fase 8: Evaluación Cualitativa

In [51]:
evaluation_records = []

def evaluate_result(result, correctness, relevance, faithfulness,
                     multi_doc_integration, insufficient_evidence_recognized, comments=""):
    """
    Registra el juicio subjetivo para un resultado de rag_pipeline().
    Los scores van de 1 a 5. insufficient_evidence_recognized es booleano.
    """
    evaluation_records.append({
        "query": result["query"],
        "correctness": correctness,
        "relevance": relevance,
        "faithfulness": faithfulness,
        "multi_doc_integration": multi_doc_integration,
        "insufficient_evidence_recognized": "Sí" if insufficient_evidence_recognized else "No",
        "comments": comments
    })

In [52]:
result = rag_pipeline(test_queries[0])
display_rag_result(result)

/tmp/ipykernel_946/2754236030.py:19: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


CONSULTA: What are the main applications of Graph Neural Networks?

RESPUESTA GENERADA:

Based on the provided evidence, the main applications of Graph Neural Networks (GNNs) include:

*   **Graph analysis tasks** (Evidence 1)
*   **Graph node classification** (Evidence 1, Evidence 4)
*   **Link prediction** (Evidence 4)
*   **Predicting graph structured data** (Evidence 5)
*   **Sentiment analysis of text data** (Evidence 5)
*   **Structure-property relationships in quantum chemistry** (Evidence 5)
*   **Image classification** (Evidence 5)

------------------------------------------------------------------------------------------
EVIDENCIAS UTILIZADAS (5):
------------------------------------------------------------------------------------------

[Evidencia 1] — Rerank score: 4.2091
Title: AutoGraph: Automated Graph Neural Network
Abstract: Graphs play an important role in many applications. Recently, Graph Neural
Networks (GNNs) have achieved promising results in graph analysis tasks

In [53]:
evaluate_result(
    result,
    correctness=5,
    relevance=5,
    faithfulness=4,
    multi_doc_integration=4,
    insufficient_evidence_recognized=False,
    comments="La respuesta cubre bien las aplicaciones principales de GNNs, integrando 2-3 papers distintos."
)

In [54]:
result_ood = rag_pipeline("What is the best recipe for a traditional Ecuadorian ceviche?")
display_rag_result(result_ood)

/tmp/ipykernel_946/2754236030.py:19: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


CONSULTA: What is the best recipe for a traditional Ecuadorian ceviche?

RESPUESTA GENERADA:

El corpus no contiene información suficiente para responder a esta consulta con certeza. Las evidencias proporcionadas tratan sobre distribuciones estadísticas, modelos de aprendizaje automático, optimización de redes neuronales y reconocimiento de imágenes de alimentos, pero no incluyen ninguna receta para el ceviche ecuatoriano tradicional.

------------------------------------------------------------------------------------------
EVIDENCIAS UTILIZADAS (5):
------------------------------------------------------------------------------------------

[Evidencia 1] — Rerank score: -11.1836
Title: Von Mises-Fisher Elliptical Distribution
Abstract: A large class of modern probabilistic learning systems assumes symmetric
distributions, however, real-world data tend to obey skewed distributions and
are thus not always adequately modelled through symmetric distributions. To
address this issue, ellipt

In [55]:
evaluate_result(
    result_ood,
    correctness=5,
    relevance=5,
    faithfulness=5,
    multi_doc_integration=1,
    insufficient_evidence_recognized=True,
    comments="El sistema reconoció correctamente que el corpus no contiene información sobre este tema."
)

In [56]:
eval_df = pd.DataFrame(evaluation_records)
eval_df

,query,correctness,relevance,faithfulness,multi_doc_integration,insufficient_evidence_recognized,comments
0,What are the main applications of Graph Neural...,5,5,4,4,No,La respuesta cubre bien las aplicaciones princ...
1,What is the best recipe for a traditional Ecua...,5,5,5,1,Sí,El sistema reconoció correctamente que el corp...


---

### Fase 9: Interfaz Web Conversacional (Streamlit)

---

**Nota:** Esta parte sirve para descargar la BD dado que el examen fue desarrollado en Google Colab

In [60]:
import os

for root, dirs, files in os.walk("arxiv.db"):
    for f in files:
        path = os.path.join(root, f)
        print(path, "-", os.path.getsize(path), "bytes")

arxiv.db/LOCK - 0 bytes
arxiv.db/collections/papers/schema.json - 952 bytes
arxiv.db/collections/papers/manifest.json - 464 bytes
arxiv.db/collections/papers/wal/wal_data_000001.arrow - 35315336 bytes


In [61]:
!rm -f arxiv.zip
!zip -r arxiv.zip arxiv.db

  adding: arxiv.db/ (stored 0%)
  adding: arxiv.db/databases/ (stored 0%)
  adding: arxiv.db/.database-staging/ (stored 0%)
  adding: arxiv.db/LOCK (stored 0%)
  adding: arxiv.db/collections/ (stored 0%)
  adding: arxiv.db/collections/papers/ (stored 0%)
  adding: arxiv.db/collections/papers/wal/ (stored 0%)
  adding: arxiv.db/collections/papers/wal/wal_data_000001.arrow (deflated 25%)
  adding: arxiv.db/collections/papers/schema.json (deflated 71%)
  adding: arxiv.db/collections/papers/manifest.json (deflated 49%)


In [62]:
import zipfile

with zipfile.ZipFile("arxiv.zip", "r") as z:
    for name in z.namelist():
        print(name)

arxiv.db/
arxiv.db/databases/
arxiv.db/.database-staging/
arxiv.db/LOCK
arxiv.db/collections/
arxiv.db/collections/papers/
arxiv.db/collections/papers/wal/
arxiv.db/collections/papers/wal/wal_data_000001.arrow
arxiv.db/collections/papers/schema.json
arxiv.db/collections/papers/manifest.json


In [63]:
from google.colab import files
files.download("arxiv.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

- Link al sitio web: https://arxiv-enriquez-ri.streamlit.app/
- Repo de la App Web: https://github.com/INub3/arxiv-rag-RI.git


---